# NetCore TAC Desk: Generate synthetic support tickets

At triage, the goal is to predict total hours from ticket creation to resolution. Start with a fixed seed so running all cells recreates the same 2,500 fictional tickets.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ticket_count = 2_500
rng = np.random.default_rng(64)
sla_hours = {"S1": 4, "S2": 24, "S3": 48, "S4": 72}

## Generate ticket details

Sample UTC creation times across 2024 and 2025. Weight the categories so critical tickets are uncommon; give higher-severity cases a wider typical device impact and faster first responses.

In [2]:
start = pd.Timestamp("2024-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")
span_minutes = int((end - start) // pd.Timedelta(minutes=1))
minute_offsets = np.sort(rng.integers(0, span_minutes, size=ticket_count))
created_at = start + pd.to_timedelta(minute_offsets, unit="min")
ticket_id = [f"TAC-{number:05d}" for number in range(1, ticket_count + 1)]

product_family = rng.choice(
    ["Switching", "Routing", "Wireless", "SD-WAN", "Firewalls"],
    size=ticket_count, p=[0.30, 0.22, 0.20, 0.16, 0.12],
)
severity_index = rng.choice(4, size=ticket_count, p=[0.06, 0.22, 0.49, 0.23])
severity = np.array(list(sla_hours))[severity_index]
customer_tier = rng.choice(
    ["Standard", "Premium", "Enterprise"], size=ticket_count, p=[0.48, 0.32, 0.20],
)
region = rng.choice(["AMER", "EMEA", "APJC"], size=ticket_count, p=[0.45, 0.32, 0.23])
channel = rng.choice(["Portal", "Email", "Phone"], size=ticket_count, p=[0.62, 0.25, 0.13])
channel = np.where((severity == "S1") & (rng.random(ticket_count) < 0.65), "Phone", channel)

typical_device_count = np.array([18, 8, 4, 2])[severity_index]
devices_affected = rng.geometric(1 / typical_device_count).clip(1, 150)
software_version_months = rng.gamma(shape=2.2, scale=10, size=ticket_count).astype(int).clip(0, 84)
prior_ticket_90_days = rng.random(ticket_count) < 0.21
engineer_experience_years = rng.gamma(shape=2.8, scale=2.5, size=ticket_count).clip(0.5, 25).round(1)

typical_response_minutes = np.array([12, 35, 85, 155])[severity_index]
tier_response_factor = np.where(customer_tier == "Enterprise", 0.75, 1.0)
first_response_minutes = np.rint(
    typical_response_minutes * tier_response_factor * rng.lognormal(0, 0.55, size=ticket_count)
).astype(int).clip(1, 720)

## Write synthetic customer comments

Combine a product-specific symptom with an impact statement. The comments describe what customers reported by triage, not how the issue was eventually fixed.

In [3]:
symptoms = {
    "Switching": ["Switch ports keep dropping", "Intermittent packet loss on uplinks", "One VLAN is unreachable"],
    "Routing": ["Routes disappear intermittently", "BGP peers keep disconnecting", "Traffic is taking an unexpected path"],
    "Wireless": ["Clients keep losing Wi-Fi", "Devices cannot join the staff network", "Wi-Fi slows down during busy periods"],
    "SD-WAN": ["Branch tunnels keep dropping", "Traffic is not failing over", "Packet loss on the branch WAN"],
    "Firewalls": ["VPN sessions keep dropping", "Expected traffic is being blocked", "Firewall logs show unexpected timeouts"],
}
impact_notes = {
    "S1": ["A critical service is down", "There is no working path for production traffic"],
    "S2": ["A production workflow is disrupted", "Only a limited workaround is available"],
    "S3": ["Service is still usable", "The issue is intermittent"],
    "S4": ["No production outage reported", "We noticed this during a routine check"],
}
customer_comments = [
    f"{rng.choice(symptoms[family])}. {rng.choice(impact_notes[level])}."
    for family, level in zip(product_family, severity, strict=True)
]

## Simulate resolution outcomes

Estimate total hours as first-response time plus variable repair time. Holding other factors fixed, more devices, older software, more escalations, a reopen, or less engineer experience adds repair time. A breach means the rounded hours **exceed** the severity's fixed ETA (S1: 4, S2: 24, S3: 48, S4: 72 hours); equality is not a breach.

In [4]:
device_scope = np.log2(devices_affected)
expected_escalations = (
    np.array([0.55, 0.38, 0.22, 0.10])[severity_index]
    + 0.08 * device_scope + 0.003 * software_version_months + 0.14 * prior_ticket_90_days
)
escalation_count = rng.poisson(expected_escalations).clip(0, 4)
reopen_probability = 0.03 + 0.055 * prior_ticket_90_days + 0.04 * escalation_count
reopened = rng.random(ticket_count) < reopen_probability
resolution_code = rng.choice(
    ["CONFIG_CHANGE", "SOFTWARE_UPDATE", "HARDWARE_REPLACED", "WORKAROUND", "GUIDANCE"],
    size=ticket_count, p=[0.36, 0.27, 0.12, 0.15, 0.10],
)

base_repair_hours = np.array([1.6, 10.0, 21.0, 33.0])[severity_index]
repair_variation = rng.lognormal(0, 0.55, size=ticket_count)
repair_multiplier = (
    1 + 0.09 * device_scope + 0.010 * software_version_months
    + 0.28 * escalation_count + 0.52 * reopened
    + 0.40 / (1 + engineer_experience_years / 4)
)
resolution_hours = np.round(first_response_minutes / 60 + base_repair_hours * repair_multiplier * repair_variation, 2)
sla_threshold = np.array([sla_hours[level] for level in severity])
sla_breached = resolution_hours > sla_threshold

## Assemble the dataset and add missing values

Leave exactly 50 first-response times blank (2%) to simulate missing records. For triage modeling, `resolution_hours` is the target. Exclude final `escalation_count`, `reopened`, `resolution_code`, and `sla_breached` as predictors: they are not known at triage. Use first-response time and engineer experience only if recorded by triage.

In [5]:
tickets = pd.DataFrame({
    "ticket_id": ticket_id,
    "created_at": created_at,
    "product_family": product_family,
    "severity": severity,
    "customer_tier": customer_tier,
    "region": region,
    "channel": channel,
    "devices_affected": devices_affected,
    "software_version_months": software_version_months,
    "prior_ticket_90_days": prior_ticket_90_days,
    "engineer_experience_years": engineer_experience_years,
    "first_response_minutes": pd.array(first_response_minutes, dtype="Int64"),
    "customer_comments": customer_comments,
    "escalation_count": escalation_count,
    "reopened": reopened,
    "resolution_code": resolution_code,
    "sla_breached": sla_breached,
    "resolution_hours": resolution_hours,
})

missing_rows = rng.choice(ticket_count, size=round(ticket_count * 0.02), replace=False)
tickets.loc[missing_rows, "first_response_minutes"] = pd.NA

## Save and preview the CSV

Run from the workspace root or the `notebooks` folder. This cell regenerates the workspace's `data/tac_tickets.csv` and prints the shape and first five rows.

In [6]:
workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent
if not (workspace_root / "pyproject.toml").is_file():
    raise FileNotFoundError("Run this notebook from the workspace root or its notebooks folder.")

csv_path = workspace_root / "data" / "tac_tickets.csv"
csv_path.parent.mkdir(exist_ok=True)
tickets.to_csv(csv_path, index=False, date_format="%Y-%m-%dT%H:%M:%SZ", lineterminator="\n")

print("Shape:", tickets.shape)
print(tickets.head(5).to_string(index=False))

Shape: (2500, 18)
ticket_id                created_at product_family severity customer_tier region channel  devices_affected  software_version_months  prior_ticket_90_days  engineer_experience_years  first_response_minutes                                                              customer_comments  escalation_count  reopened resolution_code  sla_breached  resolution_hours
TAC-00001 2024-01-01 18:09:00+00:00       Wireless       S2      Standard   AMER  Portal                 6                       15                 False                        2.5                      33 Devices cannot join the staff network. Only a limited workaround is available.                 1      True   CONFIG_CHANGE          True             34.57
TAC-00002 2024-01-02 09:55:00+00:00      Switching       S3       Premium   AMER  Portal                11                       34                 False                        4.2                      90                  Intermittent packet loss on uplinks. Ser